<a href="https://colab.research.google.com/github/OSGeoLabBp/tutorials/blob/master/hungarian/pointcloud/pointclouds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Pontfelhők feldolgozása

Ez az összeálltás a gita Műszaki térinformatika egyesület 2026. évi konferenciájának workshopjához készült. A tananyag és a használt képek a BME Általános és Felsőgeodézia Tanszék Geo4All laborjának GitHub oldaláról letölthetők.

## Bevezetés

### Pontfelhőkben tárolt adatok

A pontfelhőkben a pozíció mellett további adatok is tárolásra kerülhetnek.

### Pontfelhők tárolási formátumai

A térinformatikához hasonlóan sokféle adatformátum használatos a pontfelhő állományok tárolására. Ezek között találhatók nyílt formátumok és zárt formátumok.

A nyílt formátumok esetén az állomány belső struktúrájának leírása bárki számára elérhető és ez alapján készíthet olyan programot, mely a formátumot olvasni és írni képes. Néhány, Magyarországon elterjedt nyílt polfelhő formátum: LAS, LAZ, e57, XYZ, PLY, PCD

A zárt formátumok egyes kereskedelmi szotverekhez kötődnek, azokat csak a megfelelő szoftvercsomag megvásárlása esetén tudjuk kezelni. Ilyen formátumok például az RCS és a RCP.

Az egyes tárolási formátumok közötti választásnál mérlegelni kell a metaadatok (intenzitás, szín, osztályozás) megőrzését, a fájlméretet/tömöríthetőséget illetve a szoftver kompatibilitást.

### Pontfelhőlhők előállítási technológiái

A pontfelhők előállításának két fő technológiája létezik a lézeres távmérés (LiDAR) és a fotogrammetria (SfM). A LiDAR technológia több részterületre bontható TLS, ALS, SLAM, stb.

### Nyílt forráskódú programok pontfelhők kezelésére

*   CloudCompare - grafikus felhasználói felület és programozási
*   PDAL - programkönyvtár és parancssori eszközök
*   PCL - programkönyvtár
*   Open3D - programkönyvtár Python programokhoz
*   Potree - internetes megjelenítés



## Nyers pontfelhők előkészítése

A nyers pontfelhők jellemzően zajjal terheltek. Olyan pontok jelenhetnek benne,melyek a valóságban nincsenek illetve a pontok sűrűsége túl nagy az adott feladatunkhoz. Mindkét esetben a pontfelhőben lévő pontok közül bizonyosakat eltávolítunk. A döntési kritériumban van különbség, ami alapján egy pontot megőrzünk vagy kihagyunk.

A zajszűrésnél használt fontosabb módszerek:

* statisztikai módszerek (Statistical Outlier Removal, Radius Outlier Removal)
* geometriai módszerek (pl. RANSAC)
* PCA alapú módszerek
* mélytanulás alapú módszerek




In [2]:
!pip uninstall ipywidgets
!pip install open3d

Found existing installation: ipywidgets 7.7.1
Uninstalling ipywidgets-7.7.1:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/ipywidgets-7.7.1.dist-info/*
    /usr/local/lib/python3.12/dist-packages/ipywidgets/*
Proceed (Y/n)? Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/uninstall.py", line 106, in run
    uninstall_pathset = req.uninstall(
                        ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/req/req_install.py", line 722, in uninstall
    uninstalled_pathset.remove(auto_confirm, verbose)
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/req/req_uninstall.py", line 364, in remove
    if auto_confirm or self._allowed_to_proceed(verbose):
                       ^^^^^^^^^^^^^^^^^^^^^^^^^

### Statistical Outlier Removal (SOR)

Számítsuk ki minden egyes pontra a **k** legközelebbi szomszéd átlagos távolságát.

$d_i = \frac {1} {k} \sum_{j=1}^k || p_i - p_{ij} ||$

Számítsuk ki ezen távolságok átlagát és szórását

$\mu = \frac 1 {n} \sum_{i=1}^n d_i$

$\sigma = \sqrt \frac {\sum_{i=1}^n (d_i - \mu)^2} {n-1}$

Vizsgáljuk meg minden pontra, hogy a k legközelebbi szomszéd átlagos távolsága nagyobb-e mint a globális átlag szórással megnövelt értéke.

$d_i > \mu + \alpha \cdot \sigma$

$alpha$ általában 1-3 közé eső konstans